In [10]:
import pandas as pd
import requests, json, time
from pathlib import Path
from datetime import datetime

# ======================================================
# CONFIGURAÇÕES
# ======================================================

BASE_PATH = Path(
    r"C:\Users\126815\OneDrive - paguemenos.com.br"
    r"\ENGENHARIA - OBRAS - Documentos"
    r"\BANCO DE DADOS - PLATAFORMA FINANCEIRA"
    r"\PROJETO_PLATAFORMA_FINANCEIRA"
)

FILE_REAL = BASE_PATH / "data" / "raw" / "EXPORT_REAL.XLSX"
FILE_COMP = BASE_PATH / "data" / "raw" / "EXPORT_COMPROMIS.XLSX"
FILE_SITE = BASE_PATH / "data" / "dimensions" / "dim_site.xlsx"

URL_APPS_SCRIPT = "https://script.google.com/macros/s/AKfycbyeLMGPlTdfRNFBIb-Y_N_ZQgg6rz0r5IcW5QuNpWWSb0QRrCw13h6-WVDiBaN2jkCyjg/exec"

# BATCH_SIZE menor = menos tempo de escrita no Sheets por request
# → lock liberado antes do próximo batch chegar
BATCH_SIZE          = 100
DELAY_ENTRE_BATCHES = 4   # segundos

# ======================================================
# CONSTANTES
# ======================================================

TAREFAS_LOJAS_NOVAS = [
    "Construtora", "Gerenciadora", "Alvo K - Utilities", "Persianas", "Luminária",
    "Ar Condicionado - AC", "Linha Branca", "Projetos", "Comunicação Visual",
    "Mobiliário", "Montagem", "Frete", "Gaveta", "Permits", "CDU", "Capitalização",
    "Provisionamento", "Preservação", "Transferência"
]

TAREFAS_OUTROS = [
    "Construtora", "Gerenciadora", "Alvo K - Utilities", "Persianas", "Luminária",
    "Ar Condicionado - AC", "Linha Branca", "Projetos", "Comunicação Visual",
    "Mobiliário", "Montagem", "Frete", "Gaveta", "Permits", "CDU", "Capitalização",
    "Provisionamento", "Reforma de Loja", "Transferência"
]

TIPO_LOJAS_NOVAS = "Lojas Novas"

MATERIAIS_MOBILIARIO = {
    20398,20534,20581,20582,20583,20584,20585,20586,20587,20588,
    20589,20590,20591,20592,20593,20594,20595,20596,20597,20598,
    20599,20600,20601,20605,20606,20607,20609,20610,20611,20612,
    20613,20614,20615,20616,20617,20618,20619,20620,20621,20622,
    20623,20624,20625,20626,20629,20630,20631,20632,20633,20773,
    20774,20775,20777,20778,20781,20783,20784,20786,20788,20789,
    20790,20792,20793,20794,20795,20796,20797,20798,20799,20801,
    20806,20807,20808,20809,20814,20815,20816,20822,24742,24743,
    24745,24775,24815,24832,25087,25088,25089,25090,25132,25133,
    25427,25429,25430,25431,25432,25696,25697,25891,25892,25896,
    25898,25899,25902,25903,25906,25907,25932,25933,25934,25946,
    25949,25950,25951,25954,25955,25956,25977,26009,26010,26011,
    26012,26014,26016,26017,26018,26019,26020,26021,26022,26023,
    26024,26025,26026,26027,26028,26029,26030,26031,26032,26033,
    26035,26036,26046,26047,26048,26049,26050,26051,26052,26053,
    26054,26055,26056,26057,26058,26059,26060,26061,26062,26063,
    26161,26162,26163,26164,26165,26176,26177,26178,26180,26181,
    26182,26183,26184,26185,26186,26187,26188,26189,26190,26191,
    26192,26193,26194,26195,26196,26197,26198,26199,26210,26211,
    26232,26233,26234,26235,26242,26243,26244,26246,26247,26248,
    26257,26258,26259,26317,26326,26328,26330,26331,26332,26340,
    26345,26374,26414,26415,26433,26531,26611,26645,26664,26710,
    26711,27880,27881,27882,27883,29464,29718,29719,29720,29721,
    29724,29728,29731,29734,29735,29736,29737,29740,29741,29758,
    29759,29762,29763,29767,29786,29787,29788,29789,29790,29791,
    29792,29793,29800,29801,29802,29803,29804,29805,29806,29808,
    29809,30061,30062,30063,30398,30520,30526,30527,30531,30536,
    32101,32437,32482,32488,32527,32550,32552,32554,32555,32556,
    32557,32570,32571,32572,32573,32574,32575,32577,32578,32579,
    32580,32581,32582,32583,32584,32585,32586,32587,32588,32589,
    32590,32591,32592,32593,32598,32600,32601,32602,32603,32604,
    32605,32606,32607,32608,32611,32614,32615,32670,32677,32698,
    32733,32750,32753,32754,32761
}

MATERIAIS_GAVETA   = {24780}
MATERIAIS_MONTAGEM = {24935}
MATERIAIS_FRETE    = {20272}

REGRAS_MATERIAIS = {
    "Construtora":        {20102},
    "Gerenciadora":       {32432},
    "Permits":            {40143, 39160, 25044, 24822, 24769, 40195, 20311, 20702, 40069, 40252},
    "Linha Branca": {
        20776, 20573, 20802, 29463, 29961, 29898, 20428, 32118, 26455, 20407, 20406,
        20817, 20804, 25433, 30509, 20819, 25212, 20011, 24713, 20460, 24715,
        29512, 32660, 20454, 20476, 29511, 30429, 26147, 32597, 32596, 26146,
        39980, 39979, 29615, 39982, 39983
    },
    "Persianas":          {25992, 26706},
    "Alvo K - Utilities": {32433},
    "Comunicação Visual": {20029},
    "CDU":                {24768},
    "Provisionamento":    {40062},
    "Montagem":           {24935},
    "Frete":              {20272},
}

TIPO_NORMALIZACAO = {
    "lojas novas":        "Lojas Novas",
    "resize":             "Resize",
    "relocation":         "Relocation",
    "retrofit":           "Retrofit",
    "virada de bandeira": "Virada de Bandeira",
}

def normalizar_tipo(valor):
    if pd.isna(valor):
        return "Lojas Novas"
    return TIPO_NORMALIZACAO.get(str(valor).strip().lower(), str(valor).strip())

# ======================================================
# HELPERS
# ======================================================

def classificar_por_texto(texto):
    t = str(texto).lower()
    if "luminaria" in t:
        return "Luminária"
    if "ar cond" in t or "cortina de ar" in t:
        return "Ar Condicionado - AC"
    if "serv" in t or "projeto" in t:
        return "Projetos"
    return None

# ======================================================
# CARGA SAP
# ======================================================

def load_real():
    df = pd.read_excel(FILE_REAL)
    df = df.rename(columns={
        "Ordem":                "ordem",
        "Material":             "material",
        "Texto breve material": "texto_breve",
        "Documento de compras": "documento",
        "Valor/moeda objeto":   "valor"
    })
    df["base"] = "REAL"
    return df

def load_compromisso():
    df = pd.read_excel(FILE_COMP)
    df = df.rename(columns={
        "Ordem":                "ordem",
        "Material":             "material",
        "Denominação":          "texto_breve",
        "Nº doc.de referência": "documento",
        "Valor/moeda objeto":   "valor"
    })
    df["base"] = "COMPROMISSO"
    return df

# ======================================================
# PIPELINE PRINCIPAL
# ======================================================

def run_pipeline():
    print("🚀 Iniciando pipeline financeiro...")

    # 1. União SAP
    df = pd.concat([load_real(), load_compromisso()], ignore_index=True)

    df["valor"]    = pd.to_numeric(df["valor"],    errors="coerce").fillna(0)
    df["material"] = pd.to_numeric(df["material"], errors="coerce")
    df["documento"] = df["documento"].fillna("").astype(str)

    df["flag_pedido"] = df["documento"].str.startswith("4")
    df["flag_rc"]     = df["documento"].str.startswith("1")

    # 2. DIM_SITE
    dim_site = pd.read_excel(FILE_SITE)
    cols_base = ["ordem", "site", "responsavel", "tipo_iniciativa"]
    if "filial" in dim_site.columns:
        cols_base.append("filial")

    dim_site = dim_site[
        dim_site["responsavel"].notna() &
        (dim_site["responsavel"].astype(str).str.strip() != "")
    ][cols_base].copy()

    if "filial" not in dim_site.columns:
        dim_site["filial"] = ""

    def _limpar_filial(v):
        if pd.isna(v) or str(v).strip() == "":
            return ""
        try:
            return str(int(float(v)))
        except (ValueError, TypeError):
            return str(v).strip()

    dim_site["filial"] = dim_site["filial"].apply(_limpar_filial)
    dim_site["tipo_iniciativa"] = dim_site["tipo_iniciativa"].apply(normalizar_tipo)

    dim_site["ordem"] = pd.to_numeric(dim_site["ordem"], errors="coerce")
    df["ordem"]       = pd.to_numeric(df["ordem"],       errors="coerce")

    mapa_tipo = dim_site.set_index("ordem")["tipo_iniciativa"].to_dict()

    # 3. Classificação de tarefa
    def classificar_material(row):
        mat = row["material"]
        if mat == 20103:
            tipo = mapa_tipo.get(row["ordem"], TIPO_LOJAS_NOVAS)
            return "Preservação" if tipo == TIPO_LOJAS_NOVAS else "Reforma de Loja"
        for tarefa, mats in REGRAS_MATERIAIS.items():
            if mat in mats:
                return tarefa
        return None

    df["tarefa"] = df.apply(classificar_material, axis=1)

    mask = df["tarefa"].isna()
    df.loc[mask, "tarefa"] = df.loc[mask, "texto_breve"].apply(classificar_por_texto)

    df.loc[
        (df["base"] == "REAL") & (df["documento"].str.strip() == ""),
        "tarefa"
    ] = "Capitalização"

    # 4. Mobiliário e Gaveta
    mask_gaveta = df["material"].isin(MATERIAIS_GAVETA) & df["tarefa"].isna()
    df.loc[mask_gaveta, "tarefa"] = "Gaveta"
    print(f"   ✅ Gaveta: {mask_gaveta.sum()} linhas")

    mask_mob = df["material"].isin(MATERIAIS_MOBILIARIO) & df["tarefa"].isna()
    df.loc[mask_mob, "tarefa"] = "Mobiliário"
    print(f"   ✅ Mobiliário: {mask_mob.sum()} linhas")

    pedidos_mob = set(
        df.loc[
            df["material"].isin(MATERIAIS_MOBILIARIO | MATERIAIS_GAVETA),
            "documento"
        ].unique()
    )
    pedidos_mob.discard("")
    mask_mob_resto = df["documento"].isin(pedidos_mob) & df["tarefa"].isna()
    df.loc[mask_mob_resto, "tarefa"] = "Mobiliário"
    print(f"   ✅ Mobiliário (linhas restantes do pedido): {mask_mob_resto.sum()}")

    # 5. Consolidação financeira
    real_df  = df[df["base"] == "REAL"].groupby(["ordem", "tarefa"])["valor"].sum().rename("REAL")
    comp_df  = df[df["base"] == "COMPROMISSO"].groupby(["ordem", "tarefa"])["valor"].sum().rename("COMPROMISSO")
    flag_df  = df.groupby(["ordem", "tarefa"])["flag_pedido"].max()

    financeiro = pd.concat([real_df, comp_df, flag_df], axis=1).fillna(0).reset_index()
    financeiro["COMPROMISSO_ABERTO"] = (
        financeiro["COMPROMISSO"] - financeiro["REAL"]
    ).clip(lower=0)

    # 6. Malha completa
    malhas = []
    for _, row in dim_site.iterrows():
        tarefas = (
            TAREFAS_LOJAS_NOVAS
            if row["tipo_iniciativa"] == TIPO_LOJAS_NOVAS
            else TAREFAS_OUTROS
        )
        for tarefa in tarefas:
            malhas.append({
                "ordem":           row["ordem"],
                "site":            row["site"],
                "responsavel":     row["responsavel"],
                "tipo_iniciativa": row["tipo_iniciativa"],
                "filial":          row["filial"],
                "tarefa":          tarefa
            })

    malha = pd.DataFrame(malhas)
    fato  = malha.merge(financeiro, on=["ordem", "tarefa"], how="left")

    fato[["REAL", "COMPROMISSO", "COMPROMISSO_ABERTO"]] = (
        fato[["REAL", "COMPROMISSO", "COMPROMISSO_ABERTO"]].fillna(0)
    )
    fato["flag_pedido"] = fato["flag_pedido"].fillna(False).infer_objects(copy=False)

    # 7. Status final
    fato["status"] = "Aguardando Requisição"

    fato.loc[
        (fato["REAL"] == 0) & (fato["COMPROMISSO"] > 0) & (~fato["flag_pedido"]),
        "status"
    ] = "Requisição Criada"

    fato.loc[
        (fato["REAL"] == 0) & (fato["COMPROMISSO"] > 0) & (fato["flag_pedido"]),
        "status"
    ] = "Pedido Criado"

    fato.loc[
        (fato["REAL"] > 0) & (fato["COMPROMISSO"] > 0),
        "status"
    ] = "Pgto Parcial"

    fato.loc[
        (fato["REAL"] > 0) & (fato["COMPROMISSO"] == 0),
        "status"
    ] = "Realizado"

    fato.loc[fato["tarefa"] == "Transferência", "status"] = "Controle Manual"

    # 8. Diagnóstico
    print("\n📋 Diagnóstico — TER36-ZequinhaFre-PI:")
    amostra = fato[fato["site"].str.contains("TER36", na=False)][
        ["tarefa", "REAL", "COMPROMISSO", "COMPROMISSO_ABERTO", "status", "tipo_iniciativa"]
    ]
    if not amostra.empty:
        print(amostra.to_string(index=False))
    else:
        print("   (site não encontrado na amostra)")

    print("\n📊 Tipos de iniciativa na base:")
    print(fato["tipo_iniciativa"].value_counts().to_string())

    print(f"\n📦 Total de linhas a enviar: {len(fato)}")
    print("\n✅ Pipeline financeiro concluído\n")
    return fato

# ======================================================
# ENVIO PARA APPS SCRIPT
#
# DIAGNÓSTICO CONFIRMADO pelo debug:
#   - O redirect do Google gera um user_content_key único por request
#   - allow_redirects=True (padrão) é o comportamento correto
#   - O problema de rows:0 era lock: 200 linhas demoram mais para
#     gravar no Sheets do que o delay anterior comportava
#   - Solução: BATCH_SIZE=100 + delay=4s → lock sempre liberado a tempo
# ======================================================

def _post(payload, descricao, delay=0):
    if delay > 0:
        print(f"   ⏳ aguardando {delay}s...", flush=True)
        time.sleep(delay)

    body = json.dumps(payload, default=str, ensure_ascii=False).encode("utf-8")
    r = requests.post(
        URL_APPS_SCRIPT,
        data=body,
        headers={"Content-Type": "application/json; charset=utf-8"},
        timeout=180,
        verify=False,
        allow_redirects=True   # padrão correto — cada request tem sua própria key
    )
    texto = r.text[:120]
    print(f"   {descricao} → HTTP {r.status_code} | {texto}")

    # Alerta se rows:0 ainda ocorrer
    if r.status_code == 200 and '"rows":0' in r.text:
        print(f"   ⚠️  rows:0 — lock ainda ativo. Aumente DELAY_ENTRE_BATCHES e tente novamente.")

    return r.status_code == 200, r.text


def enviar(base):
    cols = [
        "site", "filial", "tarefa", "REAL", "COMPROMISSO",
        "COMPROMISSO_ABERTO", "status", "responsavel", "tipo_iniciativa"
    ]
    registros = base[cols].fillna("").to_dict("records")
    total     = len(registros)
    n_batches = -(-total // BATCH_SIZE)

    print(f"📤 Enviando {total} linhas em {n_batches} batches de {BATCH_SIZE} "
          f"(delay {DELAY_ENTRE_BATCHES}s entre batches)...\n")

    # ── Batch 1: limpa a aba e escreve com cabeçalho ──────────────────────
    ok, _ = _post(
        {
            "metadata": {
                "gerado_em":    datetime.now().isoformat(),
                "total_linhas": total,
                "modo":         "reset"
            },
            "financeiro_consolidado": registros[:BATCH_SIZE]
        },
        descricao=f"Batch 1/{n_batches} ({len(registros[:BATCH_SIZE])} linhas)",
        delay=0
    )
    if not ok:
        print("❌ Falha no batch 1 — envio interrompido.")
        return

    # ── Batches 2+: append ────────────────────────────────────────────────
    for i, inicio in enumerate(range(BATCH_SIZE, total, BATCH_SIZE), start=2):
        bloco = registros[inicio: inicio + BATCH_SIZE]
        ok, _ = _post(
            {
                "metadata": {
                    "gerado_em": datetime.now().isoformat(),
                    "modo":      "append"
                },
                "financeiro_consolidado": bloco
            },
            descricao=f"Batch {i}/{n_batches} ({len(bloco)} linhas)",
            delay=DELAY_ENTRE_BATCHES
        )
        if not ok:
            print(f"❌ Falha no batch {i} — envio interrompido.")
            return

    # ── Skeleton dim_contratos ────────────────────────────────────────────
    dim_pre = base[["site", "tarefa"]].copy()
    dim_pre["fornecedor"]        = ""
    dim_pre["contratado_manual"] = ""
    dim_pre["aditivo"]           = ""
    dim_pre["avi"]               = ""

    _post(
        {
            "metadata": {"gerado_em": datetime.now().isoformat()},
            "dim_contratos_skeleton": dim_pre.to_dict("records")
        },
        descricao="Skeleton dim_contratos",
        delay=DELAY_ENTRE_BATCHES
    )

    print("\n✅ Envio concluído!\n")


# ======================================================
# MAIN
# ======================================================

if __name__ == "__main__":
    base = run_pipeline()
    enviar(base)

🚀 Iniciando pipeline financeiro...
   ✅ Gaveta: 18 linhas
   ✅ Mobiliário: 1159 linhas
   ✅ Mobiliário (linhas restantes do pedido): 30

📋 Diagnóstico — TER36-ZequinhaFre-PI:
              tarefa      REAL  COMPROMISSO  COMPROMISSO_ABERTO                status tipo_iniciativa
         Construtora 981000.00      9455.00                0.00          Pgto Parcial     Lojas Novas
        Gerenciadora  33950.00         0.00                0.00             Realizado     Lojas Novas
  Alvo K - Utilities   2150.00     17377.38            15227.38          Pgto Parcial     Lojas Novas
           Persianas  17700.00         0.00                0.00             Realizado     Lojas Novas
           Luminária  23783.22         0.00                0.00             Realizado     Lojas Novas
Ar Condicionado - AC  48875.11         0.00                0.00             Realizado     Lojas Novas
        Linha Branca  15494.02     13953.58                0.00          Pgto Parcial     Lojas Novas
         

In [ ]:
import pandas as pd
import requests, json, time
from pathlib import Path
from datetime import datetime

# ======================================================
# CONFIGURAÇÕES
# ======================================================

BASE_PATH = Path(
    r"C:\Users\126815\OneDrive - paguemenos.com.br"
    r"\ENGENHARIA - OBRAS - Documentos"
    r"\BANCO DE DADOS - PLATAFORMA FINANCEIRA"
    r"\PROJETO_PLATAFORMA_FINANCEIRA"
)

FILE_REAL = BASE_PATH / "data" / "raw" / "EXPORT_REAL.XLSX"
FILE_COMP = BASE_PATH / "data" / "raw" / "EXPORT_COMPROMIS.XLSX"
FILE_SITE = BASE_PATH / "data" / "dimensions" / "dim_site.xlsx"

URL_APPS_SCRIPT = "https://script.google.com/macros/s/AKfycbyeLMGPlTdfRNFBIb-Y_N_ZQgg6rz0r5IcW5QuNpWWSb0QRrCw13h6-WVDiBaN2jkCyjg/exec"

# Otimizado: batch grande + sem delay (problema era NaN no JSON, não timing)
BATCH_SIZE          = 500
DELAY_ENTRE_BATCHES = 0

# ======================================================
# CONSTANTES
# ======================================================

TAREFAS_LOJAS_NOVAS = [
    "Construtora", "Gerenciadora", "Alvo K - Utilities", "Persianas", "Luminária",
    "Ar Condicionado - AC", "Linha Branca", "Projetos", "Comunicação Visual",
    "Mobiliário", "Montagem", "Frete", "Gaveta", "Permits", "CDU", "Capitalização",
    "Provisionamento", "Preservação", "Transferência"
]

TAREFAS_OUTROS = [
    "Construtora", "Gerenciadora", "Alvo K - Utilities", "Persianas", "Luminária",
    "Ar Condicionado - AC", "Linha Branca", "Projetos", "Comunicação Visual",
    "Mobiliário", "Montagem", "Frete", "Gaveta", "Permits", "CDU", "Capitalização",
    "Provisionamento", "Reforma de Loja", "Transferência"
]

TIPO_LOJAS_NOVAS = "Lojas Novas"

MATERIAIS_MOBILIARIO = {
    20398,20534,20581,20582,20583,20584,20585,20586,20587,20588,
    20589,20590,20591,20592,20593,20594,20595,20596,20597,20598,
    20599,20600,20601,20605,20606,20607,20609,20610,20611,20612,
    20613,20614,20615,20616,20617,20618,20619,20620,20621,20622,
    20623,20624,20625,20626,20629,20630,20631,20632,20633,20773,
    20774,20775,20777,20778,20781,20783,20784,20786,20788,20789,
    20790,20792,20793,20794,20795,20796,20797,20798,20799,20801,
    20806,20807,20808,20809,20814,20815,20816,20822,24742,24743,
    24745,24775,24815,24832,25087,25088,25089,25090,25132,25133,
    25427,25429,25430,25431,25432,25696,25697,25891,25892,25896,
    25898,25899,25902,25903,25906,25907,25932,25933,25934,25946,
    25949,25950,25951,25954,25955,25956,25977,26009,26010,26011,
    26012,26014,26016,26017,26018,26019,26020,26021,26022,26023,
    26024,26025,26026,26027,26028,26029,26030,26031,26032,26033,
    26035,26036,26046,26047,26048,26049,26050,26051,26052,26053,
    26054,26055,26056,26057,26058,26059,26060,26061,26062,26063,
    26161,26162,26163,26164,26165,26176,26177,26178,26180,26181,
    26182,26183,26184,26185,26186,26187,26188,26189,26190,26191,
    26192,26193,26194,26195,26196,26197,26198,26199,26210,26211,
    26232,26233,26234,26235,26242,26243,26244,26246,26247,26248,
    26257,26258,26259,26317,26326,26328,26330,26331,26332,26340,
    26345,26374,26414,26415,26433,26531,26611,26645,26664,26710,
    26711,27880,27881,27882,27883,29464,29718,29719,29720,29721,
    29724,29728,29731,29734,29735,29736,29737,29740,29741,29758,
    29759,29762,29763,29767,29786,29787,29788,29789,29790,29791,
    29792,29793,29800,29801,29802,29803,29804,29805,29806,29808,
    29809,30061,30062,30063,30398,30520,30526,30527,30531,30536,
    32101,32437,32482,32488,32527,32550,32552,32554,32555,32556,
    32557,32570,32571,32572,32573,32574,32575,32577,32578,32579,
    32580,32581,32582,32583,32584,32585,32586,32587,32588,32589,
    32590,32591,32592,32593,32598,32600,32601,32602,32603,32604,
    32605,32606,32607,32608,32611,32614,32615,32670,32677,32698,
    32733,32750,32753,32754,32761
}

MATERIAIS_GAVETA   = {24780}
MATERIAIS_MONTAGEM = {24935}
MATERIAIS_FRETE    = {20272}

REGRAS_MATERIAIS = {
    "Construtora":        {20102},
    "Gerenciadora":       {32432},
    "Permits":            {40143, 39160, 25044, 24822, 24769, 40195, 20311, 20702, 40069, 40252},
    "Linha Branca": {
        20776, 20573, 20802, 29463, 29961, 29898, 20428, 32118, 26455, 20407, 20406,
        20817, 20804, 25433, 30509, 20819, 25212, 20011, 24713, 20460, 24715,
        29512, 32660, 20454, 20476, 29511, 30429, 26147, 32597, 32596, 26146,
        39980, 39979, 29615, 39982, 39983
    },
    "Persianas":          {25992, 26706},
    "Alvo K - Utilities": {32433},
    "Comunicação Visual": {20029},
    "CDU":                {24768},
    "Provisionamento":    {40062},
    "Montagem":           {24935},
    "Frete":              {20272},
}

TIPO_NORMALIZACAO = {
    "lojas novas":        "Lojas Novas",
    "resize":             "Resize",
    "relocation":         "Relocation",
    "retrofit":           "Retrofit",
    "virada de bandeira": "Virada de Bandeira",
}

def normalizar_tipo(valor):
    if pd.isna(valor):
        return "Lojas Novas"
    return TIPO_NORMALIZACAO.get(str(valor).strip().lower(), str(valor).strip())

# ======================================================
# HELPERS
# ======================================================

def classificar_por_texto(texto):
    t = str(texto).lower()
    if "luminaria" in t:
        return "Luminária"
    if "ar cond" in t or "cortina de ar" in t:
        return "Ar Condicionado - AC"
    if "serv" in t or "projeto" in t:
        return "Projetos"
    return None

# ======================================================
# CARGA SAP
# ======================================================

def load_real():
    df = pd.read_excel(FILE_REAL)
    df = df.rename(columns={
        "Ordem":                "ordem",
        "Material":             "material",
        "Texto breve material": "texto_breve",
        "Documento de compras": "documento",
        "Valor/moeda objeto":   "valor"
    })
    df["base"] = "REAL"
    return df

def load_compromisso():
    df = pd.read_excel(FILE_COMP)
    df = df.rename(columns={
        "Ordem":                "ordem",
        "Material":             "material",
        "Denominação":          "texto_breve",
        "Nº doc.de referência": "documento",
        "Valor/moeda objeto":   "valor"
    })
    df["base"] = "COMPROMISSO"
    return df

# ======================================================
# PIPELINE PRINCIPAL
# ======================================================

def run_pipeline():
    print("🚀 Iniciando pipeline financeiro...")

    # 1. União SAP
    df = pd.concat([load_real(), load_compromisso()], ignore_index=True)

    df["valor"]    = pd.to_numeric(df["valor"],    errors="coerce").fillna(0)
    df["material"] = pd.to_numeric(df["material"], errors="coerce")
    df["documento"] = df["documento"].fillna("").astype(str)

    df["flag_pedido"] = df["documento"].str.startswith("4")
    df["flag_rc"]     = df["documento"].str.startswith("1")

    # 2. DIM_SITE
    dim_site = pd.read_excel(FILE_SITE)
    cols_base = ["ordem", "site", "responsavel", "tipo_iniciativa"]
    if "filial" in dim_site.columns:
        cols_base.append("filial")

    dim_site = dim_site[
        dim_site["responsavel"].notna() &
        (dim_site["responsavel"].astype(str).str.strip() != "")
    ][cols_base].copy()

    if "filial" not in dim_site.columns:
        dim_site["filial"] = ""

    def _limpar_filial(v):
        if pd.isna(v) or str(v).strip() == "":
            return ""
        try:
            return str(int(float(v)))
        except (ValueError, TypeError):
            return str(v).strip()

    dim_site["filial"] = dim_site["filial"].apply(_limpar_filial)
    dim_site["tipo_iniciativa"] = dim_site["tipo_iniciativa"].apply(normalizar_tipo)

    dim_site["ordem"] = pd.to_numeric(dim_site["ordem"], errors="coerce")
    df["ordem"]       = pd.to_numeric(df["ordem"],       errors="coerce")

    mapa_tipo = dim_site.set_index("ordem")["tipo_iniciativa"].to_dict()

    # 3. Classificação de tarefa
    def classificar_material(row):
        mat = row["material"]
        if mat == 20103:
            tipo = mapa_tipo.get(row["ordem"], TIPO_LOJAS_NOVAS)
            return "Preservação" if tipo == TIPO_LOJAS_NOVAS else "Reforma de Loja"
        for tarefa, mats in REGRAS_MATERIAIS.items():
            if mat in mats:
                return tarefa
        return None

    df["tarefa"] = df.apply(classificar_material, axis=1)

    mask = df["tarefa"].isna()
    df.loc[mask, "tarefa"] = df.loc[mask, "texto_breve"].apply(classificar_por_texto)

    df.loc[
        (df["base"] == "REAL") & (df["documento"].str.strip() == ""),
        "tarefa"
    ] = "Capitalização"

    # 4. Mobiliário e Gaveta
    mask_gaveta = df["material"].isin(MATERIAIS_GAVETA) & df["tarefa"].isna()
    df.loc[mask_gaveta, "tarefa"] = "Gaveta"
    print(f"   ✅ Gaveta: {mask_gaveta.sum()} linhas")

    mask_mob = df["material"].isin(MATERIAIS_MOBILIARIO) & df["tarefa"].isna()
    df.loc[mask_mob, "tarefa"] = "Mobiliário"
    print(f"   ✅ Mobiliário: {mask_mob.sum()} linhas")

    pedidos_mob = set(
        df.loc[
            df["material"].isin(MATERIAIS_MOBILIARIO | MATERIAIS_GAVETA),
            "documento"
        ].unique()
    )
    pedidos_mob.discard("")
    mask_mob_resto = df["documento"].isin(pedidos_mob) & df["tarefa"].isna()
    df.loc[mask_mob_resto, "tarefa"] = "Mobiliário"
    print(f"   ✅ Mobiliário (linhas restantes do pedido): {mask_mob_resto.sum()}")

    # 5. Consolidação financeira
    real_df  = df[df["base"] == "REAL"].groupby(["ordem", "tarefa"])["valor"].sum().rename("REAL")
    comp_df  = df[df["base"] == "COMPROMISSO"].groupby(["ordem", "tarefa"])["valor"].sum().rename("COMPROMISSO")
    flag_df  = df.groupby(["ordem", "tarefa"])["flag_pedido"].max()

    financeiro = pd.concat([real_df, comp_df, flag_df], axis=1).fillna(0).reset_index()
    financeiro["COMPROMISSO_ABERTO"] = (
        financeiro["COMPROMISSO"] - financeiro["REAL"]
    ).clip(lower=0)

    # 6. Malha completa
    malhas = []
    for _, row in dim_site.iterrows():
        tarefas = (
            TAREFAS_LOJAS_NOVAS
            if row["tipo_iniciativa"] == TIPO_LOJAS_NOVAS
            else TAREFAS_OUTROS
        )
        for tarefa in tarefas:
            malhas.append({
                "ordem":           row["ordem"],
                "site":            row["site"],
                "responsavel":     row["responsavel"],
                "tipo_iniciativa": row["tipo_iniciativa"],
                "filial":          row["filial"],
                "tarefa":          tarefa
            })

    malha = pd.DataFrame(malhas)
    fato  = malha.merge(financeiro, on=["ordem", "tarefa"], how="left")

    fato[["REAL", "COMPROMISSO", "COMPROMISSO_ABERTO"]] = (
        fato[["REAL", "COMPROMISSO", "COMPROMISSO_ABERTO"]].fillna(0)
    )
    fato["flag_pedido"] = fato["flag_pedido"].fillna(False).infer_objects(copy=False)

    # 7. Status final
    fato["status"] = "Aguardando Requisição"

    fato.loc[
        (fato["REAL"] == 0) & (fato["COMPROMISSO"] > 0) & (~fato["flag_pedido"]),
        "status"
    ] = "Requisição Criada"

    fato.loc[
        (fato["REAL"] == 0) & (fato["COMPROMISSO"] > 0) & (fato["flag_pedido"]),
        "status"
    ] = "Pedido Criado"

    fato.loc[
        (fato["REAL"] > 0) & (fato["COMPROMISSO"] > 0),
        "status"
    ] = "Pgto Parcial"

    fato.loc[
        (fato["REAL"] > 0) & (fato["COMPROMISSO"] == 0),
        "status"
    ] = "Realizado"

    fato.loc[fato["tarefa"] == "Transferência", "status"] = "Controle Manual"

    # 8. Diagnóstico
    print("\n📋 Diagnóstico — TER36-ZequinhaFre-PI:")
    amostra = fato[fato["site"].str.contains("TER36", na=False)][
        ["tarefa", "REAL", "COMPROMISSO", "COMPROMISSO_ABERTO", "status", "tipo_iniciativa"]
    ]
    if not amostra.empty:
        print(amostra.to_string(index=False))
    else:
        print("   (site não encontrado na amostra)")

    print("\n📊 Tipos de iniciativa na base:")
    print(fato["tipo_iniciativa"].value_counts().to_string())

    print(f"\n📦 Total de linhas a enviar: {len(fato)}")
    print("\n✅ Pipeline financeiro concluído\n")
    return fato

# ======================================================
# ENVIO PARA APPS SCRIPT
# Fix definitivo: .fillna("") antes de to_dict() elimina NaN do JSON,
# que causava JSON.parse() falhar no Apps Script → rows:0.
# Com JSON válido, não há necessidade de delay entre batches.
# ======================================================

def _post(payload, descricao, delay=0):
    if delay > 0:
        time.sleep(delay)

    body = json.dumps(payload, default=str, ensure_ascii=False).encode("utf-8")
    r = requests.post(
        URL_APPS_SCRIPT,
        data=body,
        headers={"Content-Type": "application/json; charset=utf-8"},
        timeout=180,
        verify=False,
        allow_redirects=True
    )
    print(f"   {descricao} → HTTP {r.status_code} | {r.text[:80]}")
    return r.status_code == 200, r.text


def enviar(base):
    cols = [
        "site", "filial", "tarefa", "REAL", "COMPROMISSO",
        "COMPROMISSO_ABERTO", "status", "responsavel", "tipo_iniciativa"
    ]
    # .fillna("") garante JSON válido — sem NaN que quebra o JSON.parse() no Apps Script
    registros = base[cols].fillna("").to_dict("records")
    total     = len(registros)
    n_batches = -(-total // BATCH_SIZE)

    print(f"📤 Enviando {total} linhas em {n_batches} batch(es) de {BATCH_SIZE}...\n")

    # Batch 1: reset (limpa aba + escreve cabeçalho)
    ok, _ = _post(
        {
            "metadata": {
                "gerado_em":    datetime.now().isoformat(),
                "total_linhas": total,
                "modo":         "reset"
            },
            "financeiro_consolidado": registros[:BATCH_SIZE]
        },
        descricao=f"Batch 1/{n_batches} ({min(BATCH_SIZE, total)} linhas)",
        delay=0
    )
    if not ok:
        print("❌ Falha no batch 1 — envio interrompido.")
        return

    # Batches 2+: append
    for i, inicio in enumerate(range(BATCH_SIZE, total, BATCH_SIZE), start=2):
        bloco = registros[inicio: inicio + BATCH_SIZE]
        ok, _ = _post(
            {
                "metadata": {"gerado_em": datetime.now().isoformat(), "modo": "append"},
                "financeiro_consolidado": bloco
            },
            descricao=f"Batch {i}/{n_batches} ({len(bloco)} linhas)",
            delay=DELAY_ENTRE_BATCHES
        )
        if not ok:
            print(f"❌ Falha no batch {i} — envio interrompido.")
            return

    # Skeleton dim_contratos (rows:0 é comportamento normal — não é erro)
    dim_pre = base[["site", "tarefa"]].fillna("").copy()
    dim_pre["fornecedor"]        = ""
    dim_pre["contratado_manual"] = ""
    dim_pre["aditivo"]           = ""
    dim_pre["avi"]               = ""

    _post(
        {
            "metadata": {"gerado_em": datetime.now().isoformat()},
            "dim_contratos_skeleton": dim_pre.to_dict("records")
        },
        descricao="Skeleton dim_contratos (rows:0 = normal)",
        delay=DELAY_ENTRE_BATCHES
    )

    print("\n✅ Envio concluído!\n")


# ======================================================
# MAIN
# ======================================================

if __name__ == "__main__":
    base = run_pipeline()
    enviar(base)